# From Raw Data to the 10 Global Deciles

**What is a global decile?** Everyone in the world is ranked from poorest to richest and divided into 10 groups with equal population size (10% each). Decile 1 is the poorest 10% of the world. A decile's *share* is its mean welfare divided by the sum of the mean welfare of all 10 deciles.

**Steps**
1. Load the global distribution (1,000 bins per country and year).
2. Check data quality.
3. Add population data for each country (PIP API).
4. Construct the global deciles and their shares.
6. Save the output table and create the charts.

# Cleaning data

In [179]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
import pyreadr
import requests

RDA_FILE = Path('data/comparable_blog 2.Rda')
POP_CACHE = Path('data/pop_pip.csv')
OUT_DIR = Path('data/processed')
FIG_DIR = Path('figures')
DRAFT_DIR = FIG_DIR / 'draft'
DRAFT_DIR.mkdir(exist_ok=True)
START_YEAR = 1990
END_YEAR = 2026

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

## Step 1. Load the data

The file `comparable_blog 2.Rda` contains one row for each **country, year, and bin**. Each country is divided into 1,000 equally populated bins, from poorest to richest, and `welf_b` is the welfare of each bin in dollars per person per day.

In [180]:
bins = pyreadr.read_r(RDA_FILE)['comparable_blog']
bins['code'] = bins['code'].astype(str)
bins['year'] = bins['year'].astype(int)
bins['quantile'] = bins['quantile'].astype(int)

print(f'Rows: {len(bins):,}')
print(f'Countries: {bins["code"].nunique()}  |  Years: {bins["year"].min()}-{bins["year"].max()}')
bins.head()

Rows: 10,027,000
Countries: 218  |  Years: 1981-2026


,code,year,quantile,welf_b
0,AGO,1981,1,0.251251
1,AGO,1981,2,0.251251
2,AGO,1981,3,0.251251
3,AGO,1981,4,0.262932
4,AGO,1981,5,0.287342


## Step 2. Check data quality

Before calculating anything, we check that:
- there are no missing values or duplicate rows;
- each country-year has exactly 1,000 bins;
- welfare is positive and does not decrease from one bin to the next (otherwise the bins would not be ordered);
- no country-year combination is missing.

In [181]:
bins = bins.sort_values(['code', 'year', 'quantile']).reset_index(drop=True)
per_group = bins.groupby(['code', 'year']).size()
step = bins.groupby(['code', 'year'])['welf_b'].diff()

checks = {
    'missing values': int(bins.isna().sum().sum()),
    'duplicate rows (country, year, bin)': int(bins.duplicated(['code', 'year', 'quantile']).sum()),
    'country-years with a bin count different from 1,000': int((per_group != 1000).sum()),
    'welfare less than or equal to 0': int((bins['welf_b'] <= 0).sum()),
    'bins that decrease relative to the previous bin': int((step < 0).sum()),
    'bins with the same value as the previous bin (ties)': int((step == 0).sum()),
}
display(pd.Series(checks, name='result').to_frame())

expected = pd.MultiIndex.from_product(
    [sorted(bins['code'].unique()), range(bins['year'].min(), bins['year'].max() + 1)],
    names=['code', 'year'],
)
print('Missing country-year combinations:', list(expected.difference(per_group.index)))

,result
missing values,0
"duplicate rows (country, year, bin)",0
"country-years with a bin count different from 1,000",0
welfare less than or equal to 0,0
bins that decrease relative to the previous bin,253129
bins with the same value as the previous bin (ties),38328


Missing country-year combinations: [('GRL', 2026)]


In [182]:
drops = bins.assign(prev=bins.groupby(['code', 'year'])['welf_b'].shift())
drops = drops[drops['welf_b'] < drops['prev']]
drops['rel_drop_pct'] = 100 * (1 - drops['welf_b'] / drops['prev'])

print(drops['rel_drop_pct'].describe(percentiles=[.5, .9, .99]).round(3).to_string())
print()
print('Decreasing bins by year (selected years):')
print(drops.groupby('year').size().loc[[1990, 2000, 2010, 2020, 2026]].to_string())
print()
print('Countries with the most decreasing bins:')
print(drops.groupby('code').size().sort_values(ascending=False).head(8).to_string())

count    253129.000
mean          0.404
std           2.268
min           0.000
50%           0.091
90%           0.561
99%           5.571
max          80.508

Decreasing bins by year (selected years):
year
1990    10622
2000     6783
2010     2488
2020      467
2026      224

Countries with the most decreasing bins:
code
MKD    8957
ARM    7580
ARG    6880
DOM    6652
TON    6208
NER    5737
PRY    5651
BEN    5557


**What these checks show**
- There are no missing values, duplicates, or non-positive values, and every country-year has 1,000 bins. The only missing combination is Greenland (GRL) in 2026, which is excluded from the calculation.
- About 253,000 bins (2.5%) have a lower value than the previous bin within their country. Half fall by less than 0.1%, but some fall by as much as 80%. These cases are concentrated in the 1990s and in certain countries, including MKD, ARM, and ARG. They are not corrected: global deciles rank all bins together by welfare, so the internal ordering within each country does not affect the result. 
- Ties between consecutive bins are also left unchanged.
- The only filter applied to the data is keeping the years 1990–2026.

## Step 3. Add population data

To construct global deciles, we need to know how many people each bin represents. Each bin contains 1/1,000 of its country's population, so we download national population data from the PIP API and divide it by 1,000.

The download is cached in `data/pop_pip.csv` so it does not need to be repeated.

In [183]:
if not POP_CACHE.exists():
    resp = requests.get(
        'https://api.worldbank.org/pip/v1/aux',
        params={'table': 'pop', 'format': 'csv'},
        timeout=120,
    )
    resp.raise_for_status()
    POP_CACHE.write_text(resp.text)

pop = pd.read_csv(POP_CACHE)
pop = (
    pop[pop['data_level'] == 'national']
    .rename(columns={'country_code': 'code', 'value': 'pop'})[['code', 'year', 'pop']]
)
pop = pop[pop['year'].between(START_YEAR, END_YEAR)]

print(f'{len(pop):,} population rows, {pop["code"].nunique()} countries')
pop.head()

8,066 population rows, 218 countries


,code,year,pop
39,ABW,1990,62753.0
42,ABW,1991,65896.0
45,ABW,1992,69005.0
48,ABW,1993,73685.0
51,ABW,1994,77595.0


In [184]:
data = bins[bins['year'].between(START_YEAR, END_YEAR)].merge(
    pop, on=['code', 'year'], how='left', validate='m:1'
)
data['bin_pop'] = data['pop'] / 1000

print('Rows without population data:', int(data['pop'].isna().sum()))
world_pop = data.groupby('year')['bin_pop'].sum() / 1e9
world_pop.loc[[1990, 2000, 2010, 2020, 2026]].round(2).rename('world population (billions)').to_frame()

Rows without population data: 0


,world population (billions)
year,
1990,5.30
2000,6.16
2010,7.00
2020,7.85
2026,8.28


## Step 4. Construct the global deciles

For each year:
1. combine all country bins into one list ordered by welfare;
2. accumulate the population represented by the bins, from lowest to highest welfare;
3. assign each bin to the decile corresponding to the accumulated share of the world population (0–10% = decile 1, 10–20% = decile 2, and so on);
4. calculate each decile's population-weighted mean welfare;
5. calculate its share as that mean divided by the sum of the 10 decile means.

In [185]:
data = data.sort_values(['year', 'welf_b'], kind='stable')
data['cum_pop'] = data.groupby('year')['bin_pop'].cumsum()
data['total_pop'] = data.groupby('year')['bin_pop'].transform('sum')
data['decile'] = np.minimum(np.ceil(data['cum_pop'] / (data['total_pop'] / 10)), 10).astype(int)
data['welf_x_pop'] = data['welf_b'] * data['bin_pop']

deciles = (
    data.groupby(['year', 'decile'])
    .agg(pop=('bin_pop', 'sum'), welf_x_pop=('welf_x_pop', 'sum'))
    .reset_index()
)
deciles['mean_welf'] = deciles['welf_x_pop'] / deciles['pop']
deciles['share'] = 100 * deciles['mean_welf'] / deciles.groupby('year')['mean_welf'].transform('sum')
deciles['pop_share'] = 100 * deciles['pop'] / deciles.groupby('year')['pop'].transform('sum')
deciles = deciles[['year', 'decile', 'pop', 'pop_share', 'mean_welf', 'share']]

print('Shares sum to 100 in every year:', np.allclose(deciles.groupby('year')['share'].sum(), 100))
print(f'Decile population share, % of total: minimum {deciles["pop_share"].min():.2f}, maximum {deciles["pop_share"].max():.2f}')
deciles.head(10)

Shares sum to 100 in every year: True
Decile population share, % of total: minimum 9.98, maximum 10.02


,year,decile,pop,pop_share,mean_welf,share
0,1990,1,5.294539e+08,9.991115,1.038746,0.791751
1,1990,2,5.303288e+08,10.007627,1.595440,1.216074
2,1990,3,5.299362e+08,10.000218,2.048029,1.561045
3,1990,4,5.299587e+08,10.000642,2.561929,1.952750
4,1990,5,5.294143e+08,9.990369,3.224126,2.457488
5,1990,6,5.304051e+08,10.009067,4.237213,3.229681
6,1990,7,5.298556e+08,9.998697,6.354634,4.843619
7,1990,8,5.300396e+08,10.002168,12.100901,9.223528
8,1990,9,5.298992e+08,9.999519,26.548896,20.236054
9,1990,10,5.299553e+08,10.000578,71.486098,54.488012


## Step 5. Save the table and create the charts

The table `data/processed/deciles_year.csv` is the foundation for the remaining charts: one row per year and decile, with population, mean welfare in dollars per day, and share.

## Draft visualizations

The milestone uses Visualizations 1 through 12. Exploratory alternatives are grouped below under **Ideas**.

In [186]:
deciles.to_csv(OUT_DIR / 'deciles_year.csv', index=False)

wide = deciles.pivot(index='year', columns='decile', values='share').add_prefix('D')
wide.round(3).loc[[1990, 1999, 2014, 2026]]

decile,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
year,,,,,,,,,,
1990,0.792,1.216,1.561,1.953,2.457,3.230,4.844,9.224,20.236,54.488
1999,0.874,1.411,1.798,2.202,2.707,3.445,4.739,7.763,18.175,56.885
2014,1.040,1.712,2.204,2.768,3.526,4.649,6.494,10.075,18.513,49.020
2026,0.880,1.708,2.342,2.983,3.816,5.049,7.042,10.611,18.294,47.275


### Visualization 1. Income share by global decile

One panel per decile. Each panel has its own vertical scale to show the shape of each series, so heights cannot be compared across panels. Decile 1 is shown in blue.

In [187]:
plot_df = deciles.copy()
plot_df['decile_label'] = 'Decile ' + plot_df['decile'].astype(str)
plot_df['is_d1'] = plot_df['decile'] == 1
order = [f'Decile {i}' for i in range(1, 11)]

base = alt.Chart(plot_df).encode(
    x=alt.X('year:Q', title=None, axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020])),
    y=alt.Y(
        'share:Q',
        title='Income share (%)',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(labelExpr="datum.value + '%'"),
    ),
    color=alt.condition('datum.is_d1', alt.value('#009FDA'), alt.value('#4d4d4d')),
)

chart = (
    base.mark_line(strokeWidth=2)
    .properties(width=150, height=110)
    .facet(
        facet=alt.Facet('decile_label:N', sort=order, title=None, header=alt.Header(labelFontWeight='bold')),
        columns=5,
    )
    .resolve_scale(y='independent')
    .properties(
        title=alt.TitleParams(
            'Income share by global decile, 1990-2026',
            subtitle=[
                'Each panel has its own vertical scale.',
                'Source: World Bank Group, global distribution (1,000 bins).',
            ],
        )
    )
)
chart.save(str(DRAFT_DIR / '01_deciles_small_multiples.png'), scale_factor=2)
chart

alt.FacetChart(...)

## Visualization 2. Daily welfare by decile

The first visualization shows a relative share. We now look at the absolute level: mean welfare per person per day, expressed in 2021 PPP dollars.

As in the previous chart, each decile has its own panel and shows annual changes between 1990 and 2026. Decile 1 is shown in blue; the other deciles are shown in gray. Each panel has its own vertical scale so the evolution of each group is easy to see.

In [188]:
plot_level_df = deciles.copy()
plot_level_df['decile_label'] = 'Decile ' + plot_level_df['decile'].astype(str)
plot_level_df['is_d1'] = plot_level_df['decile'] == 1

level_order = [f'Decile {i}' for i in range(1, 11)]

level_base = alt.Chart(plot_level_df).encode(
    x=alt.X('year:Q', title=None, axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020])),
    y=alt.Y(
        'mean_welf:Q',
        title='Daily welfare (2021 PPP dollars)',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(format='$,.0f'),
    ),
    color=alt.condition('datum.is_d1', alt.value('#009FDA'), alt.value('#4d4d4d')),
)

level_chart = (
    level_base.mark_line(strokeWidth=2)
    .properties(width=150, height=110)
    .facet(
        facet=alt.Facet(
            'decile_label:N',
            sort=level_order,
            title=None,
            header=alt.Header(labelFontWeight='bold'),
        ),
        columns=5,
    )
    .resolve_scale(y='independent')
    .properties(
        title=alt.TitleParams(
            'Daily welfare by global decile, 1990-2026',
            subtitle=[
                'Each panel shows the annual change for one global population decile.',
                'Each panel has its own vertical scale. Source: World Bank Group, 2021 PPP dollars.',
            ],
        )
    )
)

level_chart.save(str(DRAFT_DIR / '02_daily_welfare_by_decile.png'), scale_factor=2)
level_chart

alt.FacetChart(...)

## Visualization 3. Relative changes in mean welfare

This visualization uses the same deciles calculated above and the mean welfare variable. Each series is indexed to 100 in 1990, so the chart shows how each decile's mean welfare changes compared with that year. It includes deciles 1 to 7 and labels each series at its endpoint.

In [189]:
relative_lines = alt.Chart(plot_relative_df).mark_line(strokeWidth=2.5).encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[START_YEAR, 2030]),
        axis=alt.Axis(
            format='d',
            values=[1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025, 2030],
        ),
    ),
    y=alt.Y(
        'index_1990:Q',
        title='Index (1990 = 100)',
        scale=alt.Scale(domain=[0, 170]),
        axis=alt.Axis(values=[0, 50, 100, 150]),
    ),
    color=alt.Color(
        'decile_label:N',
        title=None,
        scale=alt.Scale(domain=relative_order, range=list(relative_colors.values())),
        legend=None,
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('decile_label:N', title='Group'),
        alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
    ],
)

relative_chart = (
    (relative_lines + adjusted_labels)
    .properties(
        width=820,
        height=500,
        title=alt.TitleParams(
            'How global deciles have changed since 1990',
            subtitle='Income-share index by global population decile, with each series set to 100 in 1990.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

relative_chart.save(str(DRAFT_DIR / '03_relative_welfare_index.png'), scale_factor=2)
relative_chart

alt.LayerChart(...)

## Ideas

The following sections contain exploratory alternatives that are not currently used in the milestone draft.

## Idea. The national bottom 10%

This visualization compares the mean welfare of the poorest 10% within India, China, the United States, and Malawi. Each country is indexed to 100 in 1990, so the chart shows whether the poorest group in each country improved at a similar pace over time.

This is a country-level comparison, not a comparison of the global deciles above.

In [190]:
country_codes = {
    'IND': 'India',
    'CHN': 'China',
    'USA': 'United States',
    'MWI': 'Malawi',
    'COD': 'Democratic Republic of the Congo',
}

country_bottom10 = (
    bins[
        bins['code'].isin(country_codes)
        & bins['year'].between(START_YEAR, END_YEAR)
        & bins['quantile'].le(100)
    ]
    .groupby(['code', 'year'], as_index=False)
    .agg(bottom10_welfare=('welf_b', 'mean'))
)
country_bottom10['country'] = country_bottom10['code'].map(country_codes)

country_base = (
    country_bottom10[country_bottom10['year'] == START_YEAR]
    .set_index('code')['bottom10_welfare']
)
country_bottom10['index_1990'] = (
    100 * country_bottom10['bottom10_welfare']
    / country_bottom10['code'].map(country_base)
)

global_bottom10 = deciles[deciles['decile'] == 1][['year', 'mean_welf']].copy()
global_base = global_bottom10.set_index('year')['mean_welf'].loc[START_YEAR]
global_bottom10['index_1990'] = 100 * global_bottom10['mean_welf'] / global_base

country_colors = {
    'India': '#D55E00',
    'China': '#0072B2',
    'United States': '#CC79A7',
    'Malawi': '#009E73',
    'Democratic Republic of the Congo': '#6A3D9A',
}
country_order = list(country_colors)

country_lines = alt.Chart(country_bottom10).mark_line(strokeWidth=3).encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
        axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
    ),
    y=alt.Y(
        'index_1990:Q',
        title='Index (1990 = 100)',
        scale=alt.Scale(zero=False),
    ),
    color=alt.Color(
        'country:N',
        title=None,
        scale=alt.Scale(domain=country_order, range=list(country_colors.values())),
        legend=alt.Legend(orient='top'),
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('country:N', title='Country'),
        alt.Tooltip('bottom10_welfare:Q', title='National bottom 10% welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
    ],
)

global_line = alt.Chart(global_bottom10).mark_line(
    color='#111111',
    strokeDash=[6, 4],
    strokeWidth=2.5,
).encode(
    x=alt.X('year:Q'),
    y=alt.Y('index_1990:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('mean_welf:Q', title='Global bottom 10% welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Global index', format='.1f'),
    ],
)

country_chart = (
    (country_lines + global_line)
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'National bottom 10% compared with the global bottom 10%',
            subtitle='All series are indexed to 100 in 1990. Dashed line: global bottom 10%.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

country_chart.save(str(FIG_DIR / 'national_bottom10_index.png'), scale_factor=2)
country_chart

alt.LayerChart(...)

## Idea. National decile 5 compared with the global decile 5

This visualization repeats the country comparison for the middle of the distribution. Solid lines show the mean welfare of decile 5 within each country; the dashed line shows the global decile 5. All series are indexed to 100 in 1990.

In [191]:
country_decile5 = (
    bins[
        bins['code'].isin(country_codes)
        & bins['year'].between(START_YEAR, END_YEAR)
        & bins['quantile'].between(401, 500)
    ]
    .groupby(['code', 'year'], as_index=False)
    .agg(decile5_welfare=('welf_b', 'mean'))
)
country_decile5['country'] = country_decile5['code'].map(country_codes)

country_decile5_base = (
    country_decile5[country_decile5['year'] == START_YEAR]
    .set_index('code')['decile5_welfare']
)
country_decile5['index_1990'] = (
    100 * country_decile5['decile5_welfare']
    / country_decile5['code'].map(country_decile5_base)
)

global_decile5 = deciles[deciles['decile'] == 5][['year', 'mean_welf']].copy()
global_decile5_base = global_decile5.set_index('year')['mean_welf'].loc[START_YEAR]
global_decile5['index_1990'] = 100 * global_decile5['mean_welf'] / global_decile5_base

country_decile5_lines = alt.Chart(country_decile5).mark_line(strokeWidth=3).encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
        axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
    ),
    y=alt.Y(
        'index_1990:Q',
        title='Index (1990 = 100)',
        scale=alt.Scale(zero=False),
    ),
    color=alt.Color(
        'country:N',
        title=None,
        scale=alt.Scale(domain=country_order, range=list(country_colors.values())),
        legend=alt.Legend(orient='top'),
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('country:N', title='Country'),
        alt.Tooltip('decile5_welfare:Q', title='National decile 5 welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
    ],
)

global_decile5_line = alt.Chart(global_decile5).mark_line(
    color='#111111',
    strokeDash=[6, 4],
    strokeWidth=2.5,
).encode(
    x=alt.X('year:Q'),
    y=alt.Y('index_1990:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('mean_welf:Q', title='Global decile 5 welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Global index', format='.1f'),
    ],
)

decile5_chart = (
    (country_decile5_lines + global_decile5_line)
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'National decile 5 compared with the global decile 5',
            subtitle='All series are indexed to 100 in 1990. Dashed line: global decile 5.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

decile5_chart.save(str(FIG_DIR / 'national_decile5_index.png'), scale_factor=2)
decile5_chart

alt.LayerChart(...)

## Idea. National decile 10 compared with the global decile 10

This visualization repeats the comparison for the richest 10% of the population. Solid lines show national decile 10; the dashed line shows global decile 10. All series are indexed to 100 in 1990.

In [192]:
country_decile10 = (
    bins[
        bins['code'].isin(country_codes)
        & bins['year'].between(START_YEAR, END_YEAR)
        & bins['quantile'].between(901, 1000)
    ]
    .groupby(['code', 'year'], as_index=False)
    .agg(decile10_welfare=('welf_b', 'mean'))
)
country_decile10['country'] = country_decile10['code'].map(country_codes)

country_decile10_base = (
    country_decile10[country_decile10['year'] == START_YEAR]
    .set_index('code')['decile10_welfare']
)
country_decile10['index_1990'] = (
    100 * country_decile10['decile10_welfare']
    / country_decile10['code'].map(country_decile10_base)
)

global_decile10 = deciles[deciles['decile'] == 10][['year', 'mean_welf']].copy()
global_decile10_base = global_decile10.set_index('year')['mean_welf'].loc[START_YEAR]
global_decile10['index_1990'] = 100 * global_decile10['mean_welf'] / global_decile10_base

country_decile10_lines = alt.Chart(country_decile10).mark_line(strokeWidth=3).encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
        axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
    ),
    y=alt.Y(
        'index_1990:Q',
        title='Index (1990 = 100)',
        scale=alt.Scale(zero=False),
    ),
    color=alt.Color(
        'country:N',
        title=None,
        scale=alt.Scale(domain=country_order, range=list(country_colors.values())),
        legend=alt.Legend(orient='top'),
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('country:N', title='Country'),
        alt.Tooltip('decile10_welfare:Q', title='National decile 10 welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
    ],
)

global_decile10_line = alt.Chart(global_decile10).mark_line(
    color='#111111',
    strokeDash=[6, 4],
    strokeWidth=2.5,
).encode(
    x=alt.X('year:Q'),
    y=alt.Y('index_1990:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('mean_welf:Q', title='Global decile 10 welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Global index', format='.1f'),
    ],
)

decile10_chart = (
    (country_decile10_lines + global_decile10_line)
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'National decile 10 compared with the global decile 10',
            subtitle='All series are indexed to 100 in 1990. Dashed line: global decile 10.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

decile10_chart.save(str(FIG_DIR / 'national_decile10_index.png'), scale_factor=2)
decile10_chart

alt.LayerChart(...)

## Idea. National decile 2 compared with the global decile 2

This visualization compares the second decile within each country with the global second decile. Solid lines show national decile 2; the dashed line shows global decile 2. All series are indexed to 100 in 1990.

In [193]:
country_decile2 = (
    bins[
        bins['code'].isin(country_codes)
        & bins['year'].between(START_YEAR, END_YEAR)
        & bins['quantile'].between(101, 200)
    ]
    .groupby(['code', 'year'], as_index=False)
    .agg(decile2_welfare=('welf_b', 'mean'))
)
country_decile2['country'] = country_decile2['code'].map(country_codes)

country_decile2_base = (
    country_decile2[country_decile2['year'] == START_YEAR]
    .set_index('code')['decile2_welfare']
)
country_decile2['index_1990'] = (
    100 * country_decile2['decile2_welfare']
    / country_decile2['code'].map(country_decile2_base)
)

global_decile2 = deciles[deciles['decile'] == 2][['year', 'mean_welf']].copy()
global_decile2_base = global_decile2.set_index('year')['mean_welf'].loc[START_YEAR]
global_decile2['index_1990'] = 100 * global_decile2['mean_welf'] / global_decile2_base

country_decile2_lines = alt.Chart(country_decile2).mark_line(strokeWidth=3).encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
        axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
    ),
    y=alt.Y(
        'index_1990:Q',
        title='Index (1990 = 100)',
        scale=alt.Scale(zero=False),
    ),
    color=alt.Color(
        'country:N',
        title=None,
        scale=alt.Scale(domain=country_order, range=list(country_colors.values())),
        legend=alt.Legend(orient='top'),
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('country:N', title='Country'),
        alt.Tooltip('decile2_welfare:Q', title='National decile 2 welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
    ],
)

global_decile2_line = alt.Chart(global_decile2).mark_line(
    color='#111111',
    strokeDash=[6, 4],
    strokeWidth=2.5,
).encode(
    x=alt.X('year:Q'),
    y=alt.Y('index_1990:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('mean_welf:Q', title='Global decile 2 welfare', format='.2f'),
        alt.Tooltip('index_1990:Q', title='Global index', format='.1f'),
    ],
)

decile2_chart = (
    (country_decile2_lines + global_decile2_line)
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'National decile 2 compared with the global decile 2',
            subtitle='All series are indexed to 100 in 1990. Dashed line: global decile 2.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

decile2_chart.save(str(FIG_DIR / 'national_decile2_index.png'), scale_factor=2)
decile2_chart

alt.LayerChart(...)

## Idea. Malawi deciles compared with the corresponding global deciles

This visualization shows Malawi's ten deciles separately. In each panel, the solid line is Malawi and the dashed line is the corresponding global decile. All series are indexed to 100 in 1990.

In [194]:
malawi_deciles = (
    bins[
        (bins['code'] == 'MWI')
        & bins['year'].between(START_YEAR, END_YEAR)
    ]
    .assign(decile=lambda frame: ((frame['quantile'] - 1) // 100) + 1)
    .groupby(['decile', 'year'], as_index=False)
    .agg(welfare=('welf_b', 'mean'))
)
malawi_decile_base = (
    malawi_deciles[malawi_deciles['year'] == START_YEAR]
    .set_index('decile')['welfare']
)
malawi_deciles['index_1990'] = (
    100 * malawi_deciles['welfare']
    / malawi_deciles['decile'].map(malawi_decile_base)
)
malawi_deciles['series'] = 'Malawi'

global_deciles_by_group = deciles[['decile', 'year', 'mean_welf']].copy()
global_decile_base = (
    global_deciles_by_group[global_deciles_by_group['year'] == START_YEAR]
    .set_index('decile')['mean_welf']
)
global_deciles_by_group['index_1990'] = (
    100 * global_deciles_by_group['mean_welf']
    / global_deciles_by_group['decile'].map(global_decile_base)
)
global_deciles_by_group['series'] = 'Global corresponding decile'
global_deciles_by_group = global_deciles_by_group.rename(columns={'mean_welf': 'welfare'})

malawi_global_deciles = pd.concat(
    [malawi_deciles, global_deciles_by_group],
    ignore_index=True,
)

malawi_deciles_chart = (
    alt.Chart(malawi_global_deciles)
    .mark_line()
    .properties(width=760, height=75)
    .encode(
        x=alt.X(
            'year:Q',
            title=None,
            scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
            axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
        ),
        y=alt.Y(
            'index_1990:Q',
            title='Index (1990 = 100)',
            scale=alt.Scale(zero=False),
        ),
        color=alt.Color(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Malawi', 'Global corresponding decile'],
                range=['#009E73', '#111111'],
            ),
            legend=alt.Legend(orient='top'),
        ),
        strokeDash=alt.StrokeDash(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Malawi', 'Global corresponding decile'],
                range=[[1, 0], [6, 4]],
            ),
            legend=None,
        ),
        detail='series:N',
        tooltip=[
            alt.Tooltip('decile:O', title='Decile'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('series:N', title='Series'),
            alt.Tooltip('welfare:Q', title='Welfare', format='.2f'),
            alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
        ],
    )
    .facet(
        row=alt.Row(
            'decile:O',
            title=None,
            sort=list(range(1, 11)),
            header=alt.Header(labelExpr="'Decile ' + datum.value", labelAngle=0),
        ),
    )
    .properties(
        title=alt.TitleParams(
            'Malawi deciles compared with the corresponding global deciles',
            subtitle='Solid green line: Malawi. Dashed black line: global corresponding decile. All series are indexed to 100 in 1990.',
        ),
    )
    .resolve_scale(y='shared')
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

malawi_deciles_chart.save(str(DRAFT_DIR / '04_malawi_deciles_vs_global.png'), scale_factor=2)
malawi_deciles_chart

alt.FacetChart(...)

## Idea. Rwanda deciles compared with the corresponding global deciles

This visualization shows Rwanda's ten deciles separately. In each panel, the solid line is Rwanda and the dashed line is the corresponding global decile. All series are indexed to 100 in 1990.

In [195]:
rwanda_deciles = (
    bins[
        (bins['code'] == 'RWA')
        & bins['year'].between(START_YEAR, END_YEAR)
    ]
    .assign(decile=lambda frame: ((frame['quantile'] - 1) // 100) + 1)
    .groupby(['decile', 'year'], as_index=False)
    .agg(welfare=('welf_b', 'mean'))
)
rwanda_decile_base = (
    rwanda_deciles[rwanda_deciles['year'] == START_YEAR]
    .set_index('decile')['welfare']
)
rwanda_deciles['index_1990'] = (
    100 * rwanda_deciles['welfare']
    / rwanda_deciles['decile'].map(rwanda_decile_base)
)
rwanda_deciles['series'] = 'Rwanda'

global_deciles_rwanda = deciles[['decile', 'year', 'mean_welf']].copy()
global_decile_base_rwanda = (
    global_deciles_rwanda[global_deciles_rwanda['year'] == START_YEAR]
    .set_index('decile')['mean_welf']
)
global_deciles_rwanda['index_1990'] = (
    100 * global_deciles_rwanda['mean_welf']
    / global_deciles_rwanda['decile'].map(global_decile_base_rwanda)
)
global_deciles_rwanda['series'] = 'Global corresponding decile'
global_deciles_rwanda = global_deciles_rwanda.rename(columns={'mean_welf': 'welfare'})

rwanda_global_deciles = pd.concat(
    [rwanda_deciles, global_deciles_rwanda],
    ignore_index=True,
)

rwanda_deciles_chart = (
    alt.Chart(rwanda_global_deciles)
    .mark_line()
    .properties(width=760, height=75)
    .encode(
        x=alt.X(
            'year:Q',
            title=None,
            scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
            axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
        ),
        y=alt.Y(
            'index_1990:Q',
            title='Index (1990 = 100)',
            scale=alt.Scale(zero=False),
        ),
        color=alt.Color(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Rwanda', 'Global corresponding decile'],
                range=['#C44E52', '#111111'],
            ),
            legend=alt.Legend(orient='top'),
        ),
        strokeDash=alt.StrokeDash(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Rwanda', 'Global corresponding decile'],
                range=[[1, 0], [6, 4]],
            ),
            legend=None,
        ),
        detail='series:N',
        tooltip=[
            alt.Tooltip('decile:O', title='Decile'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('series:N', title='Series'),
            alt.Tooltip('welfare:Q', title='Welfare', format='.2f'),
            alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
        ],
    )
    .facet(
        row=alt.Row(
            'decile:O',
            title=None,
            sort=list(range(1, 11)),
            header=alt.Header(labelExpr="'Decile ' + datum.value", labelAngle=0),
        ),
    )
    .properties(
        title=alt.TitleParams(
            'Rwanda deciles compared with the corresponding global deciles',
            subtitle='Solid red line: Rwanda. Dashed black line: global corresponding decile. All series are indexed to 100 in 1990.',
        ),
    )
    .resolve_scale(y='shared')
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

rwanda_deciles_chart.save(str(DRAFT_DIR / '05_rwanda_deciles_vs_global.png'), scale_factor=2)
rwanda_deciles_chart

alt.FacetChart(...)

## Idea. Norway deciles compared with the corresponding global deciles

This visualization shows Norway's ten deciles separately. In each panel, the solid line is Norway and the dashed line is the corresponding global decile. All series are indexed to 100 in 1990.

In [196]:
norway_deciles = (
    bins[
        (bins['code'] == 'NOR')
        & bins['year'].between(START_YEAR, END_YEAR)
    ]
    .assign(decile=lambda frame: ((frame['quantile'] - 1) // 100) + 1)
    .groupby(['decile', 'year'], as_index=False)
    .agg(welfare=('welf_b', 'mean'))
)
norway_decile_base = (
    norway_deciles[norway_deciles['year'] == START_YEAR]
    .set_index('decile')['welfare']
)
norway_deciles['index_1990'] = (
    100 * norway_deciles['welfare']
    / norway_deciles['decile'].map(norway_decile_base)
)
norway_deciles['series'] = 'Norway'

global_deciles_norway = deciles[['decile', 'year', 'mean_welf']].copy()
global_decile_base_norway = (
    global_deciles_norway[global_deciles_norway['year'] == START_YEAR]
    .set_index('decile')['mean_welf']
)
global_deciles_norway['index_1990'] = (
    100 * global_deciles_norway['mean_welf']
    / global_deciles_norway['decile'].map(global_decile_base_norway)
)
global_deciles_norway['series'] = 'Global corresponding decile'
global_deciles_norway = global_deciles_norway.rename(columns={'mean_welf': 'welfare'})

norway_global_deciles = pd.concat(
    [norway_deciles, global_deciles_norway],
    ignore_index=True,
)

norway_deciles_chart = (
    alt.Chart(norway_global_deciles)
    .mark_line()
    .properties(width=760, height=75)
    .encode(
        x=alt.X(
            'year:Q',
            title=None,
            scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
            axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
        ),
        y=alt.Y(
            'index_1990:Q',
            title='Index (1990 = 100)',
            scale=alt.Scale(zero=False),
        ),
        color=alt.Color(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Norway', 'Global corresponding decile'],
                range=['#E69F00', '#111111'],
            ),
            legend=alt.Legend(orient='top'),
        ),
        strokeDash=alt.StrokeDash(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Norway', 'Global corresponding decile'],
                range=[[1, 0], [6, 4]],
            ),
            legend=None,
        ),
        detail='series:N',
        tooltip=[
            alt.Tooltip('decile:O', title='Decile'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('series:N', title='Series'),
            alt.Tooltip('welfare:Q', title='Welfare', format='.2f'),
            alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
        ],
    )
    .facet(
        row=alt.Row(
            'decile:O',
            title=None,
            sort=list(range(1, 11)),
            header=alt.Header(labelExpr="'Decile ' + datum.value", labelAngle=0),
        ),
    )
    .properties(
        title=alt.TitleParams(
            'Norway deciles compared with the corresponding global deciles',
            subtitle='Solid orange line: Norway. Dashed black line: global corresponding decile. All series are indexed to 100 in 1990.',
        ),
    )
    .resolve_scale(y='shared')
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

norway_deciles_chart.save(str(FIG_DIR / 'norway_deciles_vs_global.png'), scale_factor=2)
norway_deciles_chart

alt.FacetChart(...)

## Idea. Chile deciles compared with the corresponding global deciles

This visualization shows Chile's ten deciles separately. In each panel, the solid line is Chile and the dashed line is the corresponding global decile. All series are indexed to 100 in 1990.

In [197]:
chile_deciles = (
    bins[
        (bins['code'] == 'CHL')
        & bins['year'].between(START_YEAR, END_YEAR)
    ]
    .assign(decile=lambda frame: ((frame['quantile'] - 1) // 100) + 1)
    .groupby(['decile', 'year'], as_index=False)
    .agg(welfare=('welf_b', 'mean'))
)
chile_decile_base = (
    chile_deciles[chile_deciles['year'] == START_YEAR]
    .set_index('decile')['welfare']
)
chile_deciles['index_1990'] = (
    100 * chile_deciles['welfare']
    / chile_deciles['decile'].map(chile_decile_base)
)
chile_deciles['series'] = 'Chile'

global_deciles_chile = deciles[['decile', 'year', 'mean_welf']].copy()
global_decile_base_chile = (
    global_deciles_chile[global_deciles_chile['year'] == START_YEAR]
    .set_index('decile')['mean_welf']
)
global_deciles_chile['index_1990'] = (
    100 * global_deciles_chile['mean_welf']
    / global_deciles_chile['decile'].map(global_decile_base_chile)
)
global_deciles_chile['series'] = 'Global corresponding decile'
global_deciles_chile = global_deciles_chile.rename(columns={'mean_welf': 'welfare'})

chile_global_deciles = pd.concat(
    [chile_deciles, global_deciles_chile],
    ignore_index=True,
)

chile_deciles_chart = (
    alt.Chart(chile_global_deciles)
    .mark_line()
    .properties(width=760, height=75)
    .encode(
        x=alt.X(
            'year:Q',
            title=None,
            scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
            axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026]),
        ),
        y=alt.Y(
            'index_1990:Q',
            title='Index (1990 = 100)',
            scale=alt.Scale(zero=False),
        ),
        color=alt.Color(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Chile', 'Global corresponding decile'],
                range=['#0072B2', '#111111'],
            ),
            legend=alt.Legend(orient='top'),
        ),
        strokeDash=alt.StrokeDash(
            'series:N',
            title=None,
            scale=alt.Scale(
                domain=['Chile', 'Global corresponding decile'],
                range=[[1, 0], [6, 4]],
            ),
            legend=None,
        ),
        detail='series:N',
        tooltip=[
            alt.Tooltip('decile:O', title='Decile'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('series:N', title='Series'),
            alt.Tooltip('welfare:Q', title='Welfare', format='.2f'),
            alt.Tooltip('index_1990:Q', title='Index', format='.1f'),
        ],
    )
    .facet(
        row=alt.Row(
            'decile:O',
            title=None,
            sort=list(range(1, 11)),
            header=alt.Header(labelExpr="'Decile ' + datum.value", labelAngle=0),
        ),
    )
    .properties(
        title=alt.TitleParams(
            'Chile deciles compared with the corresponding global deciles',
            subtitle='Solid blue line: Chile. Dashed black line: global corresponding decile. All series are indexed to 100 in 1990.',
        ),
    )
    .resolve_scale(y='shared')
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)

chile_deciles_chart.save(str(FIG_DIR / 'chile_deciles_vs_global.png'), scale_factor=2)
chile_deciles_chart

alt.FacetChart(...)

## Draft visualizations 6 and 7. Growth and inequality

These four global visualizations focus on the distribution as a whole: income shares, growth by decile, Lorenz curves, and the welfare gap between the richest and poorest deciles.

In [198]:
# Idea: global income shares as a composition
share_area = alt.Chart(deciles).mark_area().encode(
    x=alt.X('year:Q', title=None, axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026])),
    y=alt.Y('share:Q', title='Global income share (%)', stack='zero', scale=alt.Scale(domain=[0, 100])),
    color=alt.Color('decile:N', title='Global decile', scale=alt.Scale(scheme='blues')),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('decile:O', title='Decile'),
        alt.Tooltip('share:Q', title='Income share (%)', format='.2f'),
    ],
)
share_area_chart = (
    share_area.properties(
        width=820,
        height=420,
        title=alt.TitleParams('Global income shares by decile', subtitle='The stacked areas sum to 100% in every year.'),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
share_area_chart.save(str(FIG_DIR / 'global_income_shares_area.png')), scale_factor=2)

# Visualization 6: global growth incidence curve
welfare_wide = deciles.pivot(index='decile', columns='year', values='mean_welf')
gic = welfare_wide.reset_index()[['decile', START_YEAR, END_YEAR]].copy()
gic = gic.rename(columns={START_YEAR: 'welfare_start', END_YEAR: 'welfare_end'})
gic['growth_pct'] = 100 * (gic['welfare_end'] / gic['welfare_start'] - 1)
average_growth_pct = 100 * (
    deciles.loc[deciles['year'] == END_YEAR, 'mean_welf'].mean()
    / deciles.loc[deciles['year'] == START_YEAR, 'mean_welf'].mean()
    - 1
)
gic['above_average'] = gic['growth_pct'] >= average_growth_pct
gic_bars = alt.Chart(gic).mark_bar().encode(
    x=alt.X('decile:O', title='Global decile'),
    y=alt.Y('growth_pct:Q', title=f'Welfare growth, {START_YEAR}-{END_YEAR} (%)'),
    color=alt.condition('datum.above_average', alt.value('#009E73'), alt.value('#4d4d4d')),
    tooltip=[
        alt.Tooltip('decile:O', title='Decile'),
        alt.Tooltip('growth_pct:Q', title='Growth (%)', format='.1f'),
        alt.Tooltip('above_average:N', title='At or above global average'),
    ],
)
gic_average_rule = alt.Chart(pd.DataFrame({'average_growth_pct': [average_growth_pct]})).mark_rule(
    color='#D55E00', strokeDash=[6, 4], strokeWidth=2,
).encode(
    y=alt.Y('average_growth_pct:Q'),
    tooltip=[alt.Tooltip('average_growth_pct:Q', title='Average growth (%)', format='.1f')],
)
gic_chart = (
    (gic_bars + gic_average_rule)
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams('Global Growth Incidence Curve', subtitle='Bars show welfare growth by global decile. Dashed line: growth of mean global welfare.'),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
gic_chart.save(str(DRAFT_DIR / '06_global_growth_incidence_curve.png'), scale_factor=2)

# Idea: global Lorenz curves
lorenz_parts = []
for selected_year in [START_YEAR, 2000, 2010, END_YEAR]:
    year_deciles = deciles[deciles['year'] == selected_year].sort_values('decile').copy()
    year_deciles['cum_population'] = year_deciles['pop_share'].cumsum()
    year_deciles['cum_income'] = year_deciles['share'].cumsum()
    year_deciles = pd.concat([
        pd.DataFrame({'year': [selected_year], 'cum_population': [0.0], 'cum_income': [0.0]}),
        year_deciles[['year', 'cum_population', 'cum_income']],
    ], ignore_index=True)
    lorenz_parts.append(year_deciles)
lorenz = pd.concat(lorenz_parts, ignore_index=True)
equality_line = pd.DataFrame({'cum_population': [0, 100], 'cum_income': [0, 100]})
lorenz_lines = alt.Chart(lorenz).mark_line(strokeWidth=3).encode(
    x=alt.X('cum_population:Q', title='Cumulative population share (%)'),
    y=alt.Y('cum_income:Q', title='Cumulative income share (%)'),
    color=alt.Color('year:O', title='Year', scale=alt.Scale(scheme='viridis')),
    detail='year:O',
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('cum_population:Q', title='Population (%)', format='.1f'),
        alt.Tooltip('cum_income:Q', title='Income (%)', format='.1f'),
    ],
)
equality_reference = alt.Chart(equality_line).mark_line(
    color='#111111', strokeDash=[6, 4], strokeWidth=1.5,
).encode(x='cum_population:Q', y='cum_income:Q')
lorenz_chart = (
    (lorenz_lines + equality_reference)
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams('Global Lorenz curves', subtitle='Curves farther below the equality line indicate greater global inequality.'),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
lorenz_chart.save(str(FIG_DIR / 'global_lorenz_curves.png'), scale_factor=2)

# Visualization 7: welfare ratio between the richest and poorest global deciles
extremes = deciles.pivot(index='year', columns='decile', values='mean_welf').reset_index()
extremes.columns = [str(column) for column in extremes.columns]
extremes['welfare_ratio_d10_d1'] = extremes['10'] / extremes['1']
ratio_chart = alt.Chart(extremes).mark_line(color='#6A3D9A', strokeWidth=3).encode(
    x=alt.X('year:Q', title=None, axis=alt.Axis(format='d', values=[1990, 2000, 2010, 2020, 2026])),
    y=alt.Y('welfare_ratio_d10_d1:Q', title='D10 / D1 welfare ratio', scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('welfare_ratio_d10_d1:Q', title='D10 / D1 ratio', format='.1f'),
    ],
)
ratio_chart = (
    ratio_chart.properties(
        width=820,
        height=420,
        title=alt.TitleParams('Global welfare gap between the richest and poorest deciles', subtitle='Ratio of mean welfare in decile 10 to mean welfare in decile 1.'),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
ratio_chart.save(str(DRAFT_DIR / '07_global_d10_d1_welfare_ratio.png'), scale_factor=2)

share_area_chart, gic_chart, lorenz_chart, ratio_chart

SyntaxError: unmatched ')' (253214879.py, line 21)

In [ ]:
# Idea: global income shares as a composition
display(share_area_chart)

alt.Chart(...)

In [ ]:
# Visualization 6: global growth incidence curve
display(gic_chart)

alt.LayerChart(...)

In [ ]:
# Idea: global Lorenz curves
display(lorenz_chart)

alt.LayerChart(...)

In [ ]:
# Visualization 7: welfare ratio between the richest and poorest global deciles
display(ratio_chart)

alt.Chart(...)

In [ ]:
# Idea: global poorest and richest deciles over time
selected_years = [START_YEAR, 2000, 2010, 2020, END_YEAR]
global_extremes = (
    deciles[
        deciles['year'].isin(selected_years)
        & deciles['decile'].isin([1, 10])
    ]
    .loc[:, ['year', 'decile', 'mean_welf']]
    .rename(columns={'mean_welf': 'welfare'})
    .copy()
)
global_extremes['decile_label'] = global_extremes['decile'].map({1: 'Poorest 10%', 10: 'Richest 10%'})
global_extremes['year_label'] = global_extremes['year'].astype(str)
year_order = [str(year) for year in selected_years]

global_dumbbell_rules = alt.Chart(global_extremes).mark_rule(
    color='#bdbdbd',
    strokeWidth=3,
).encode(
    x=alt.X('welfare:Q', title='Mean welfare per person per day'),
    y=alt.Y(
        'year_label:N',
        title=None,
        sort=year_order,
        axis=alt.Axis(labelFontSize=13),
    ),
    detail='year:O',
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('welfare:Q', title='Mean welfare', format='.2f'),
    ],
)
global_dumbbell_points = alt.Chart(global_extremes).mark_point(size=170, filled=True).encode(
    x=alt.X('welfare:Q', title='Mean welfare per person per day'),
    y=alt.Y('year_label:N', title=None, sort=year_order),
    color=alt.Color(
        'decile_label:N',
        title=None,
        scale=alt.Scale(
            domain=['Poorest 10%', 'Richest 10%'],
            range=['#D55E00', '#0072B2'],
        ),
        legend=alt.Legend(orient='top'),
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('decile_label:N', title='Group'),
        alt.Tooltip('welfare:Q', title='Mean welfare', format='.2f'),
    ],
)
global_dumbbell_chart = (
    (global_dumbbell_rules + global_dumbbell_points)
    .properties(
        width=820,
        height=300,
        title=alt.TitleParams(
            'Global welfare of the poorest and richest 10%',
            subtitle='Each row shows the mean welfare gap between decile 1 and decile 10 in a selected year.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
global_dumbbell_chart.save(str(FIG_DIR / 'global_poorest_richest_deciles.png'), scale_factor=2)
global_dumbbell_chart

alt.LayerChart(...)

In [ ]:
# Idea: who captured the benefits of global welfare growth?
growth_by_decile = (
    deciles[
        deciles['year'].isin([START_YEAR, END_YEAR])
    ]
    .pivot(index='decile', columns='year', values='mean_welf')
    .reset_index()
    .rename(columns={START_YEAR: 'welfare_start', END_YEAR: 'welfare_end'})
)
income_share_by_decile = (
    deciles[
        deciles['year'].isin([START_YEAR, END_YEAR])
    ]
    .pivot(index='decile', columns='year', values='share')
    .reset_index()
    .rename(columns={START_YEAR: 'share_start', END_YEAR: 'share_end'})
)
growth_by_decile = growth_by_decile.merge(income_share_by_decile, on='decile', validate='one_to_one')
growth_by_decile['welfare_gain'] = growth_by_decile['welfare_end'] - growth_by_decile['welfare_start']
growth_by_decile['share_change'] = growth_by_decile['share_end'] - growth_by_decile['share_start']
growth_by_decile['decile_label'] = 'Decile ' + growth_by_decile['decile'].astype(str)
growth_by_decile['bar_color'] = np.select(
    [growth_by_decile['decile'].isin([1, 2]), growth_by_decile['decile'].eq(10)],
    ['#D55E00', '#0072B2'],
    default='#9e9e9e',
)

gain_bars = alt.Chart(growth_by_decile).mark_bar().encode(
    x=alt.X('decile_label:N', title='Global decile', sort='-x'),
    y=alt.Y('welfare_gain:Q', title='Increase in mean welfare'),
    color=alt.Color('bar_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('decile:O', title='Decile'),
        alt.Tooltip('welfare_start:Q', title=f'Welfare in {START_YEAR}', format='.2f'),
        alt.Tooltip('welfare_end:Q', title=f'Welfare in {END_YEAR}', format='.2f'),
        alt.Tooltip('welfare_gain:Q', title='Absolute gain', format='.2f'),
    ],
)
gain_chart = gain_bars.properties(
    width=390,
    height=360,
    title=alt.TitleParams(
        'Absolute welfare gains by global decile',
        subtitle=f'Change in mean welfare, {START_YEAR}-{END_YEAR}.',
    ),
)

share_change_bars = alt.Chart(growth_by_decile).mark_bar().encode(
    x=alt.X('decile_label:N', title='Global decile', sort='-x'),
    y=alt.Y('share_change:Q', title='Change in income share (percentage points)'),
    color=alt.Color('bar_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('decile:O', title='Decile'),
        alt.Tooltip('share_start:Q', title=f'Income share in {START_YEAR}', format='.2f'),
        alt.Tooltip('share_end:Q', title=f'Income share in {END_YEAR}', format='.2f'),
        alt.Tooltip('share_change:Q', title='Change (percentage points)', format='+.2f'),
    ],
)
zero_share_rule = alt.Chart(pd.DataFrame({'zero': [0]})).mark_rule(
    color='#111111',
    strokeDash=[6, 4],
    strokeWidth=2,
).encode(
    y=alt.Y('zero:Q'),
)
share_change_chart = (share_change_bars + zero_share_rule).properties(
    width=390,
    height=360,
    title=alt.TitleParams(
        'Change in income share by global decile',
        subtitle=f'Change in percentage points, {START_YEAR}-{END_YEAR}. Dashed line: no change.',
    ),
)

growth_benefits_chart = (
    alt.hconcat(gain_chart, share_change_chart)
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
growth_benefits_chart.save(str(FIG_DIR / 'global_growth_benefits_by_decile.png'), scale_factor=2)
growth_benefits_chart

alt.HConcatChart(...)

In [ ]:
# Visualization 8: change in income share for deciles 1 to 8
growth_bottom8 = growth_by_decile[growth_by_decile['decile'].le(8)].copy()
share_change_bottom8_bars = alt.Chart(growth_bottom8).mark_bar().encode(
    x=alt.X('decile_label:N', title='Global decile', sort='-x'),
    y=alt.Y('share_change:Q', title='Change in income share (percentage points)'),
    color=alt.Color('bar_color:N', scale=None, legend=None),
    tooltip=[
        alt.Tooltip('decile:O', title='Decile'),
        alt.Tooltip('share_start:Q', title=f'Income share in {START_YEAR}', format='.2f'),
        alt.Tooltip('share_end:Q', title=f'Income share in {END_YEAR}', format='.2f'),
        alt.Tooltip('share_change:Q', title='Change (percentage points)', format='+.2f'),
    ],
)
share_change_bottom8_chart = (
    share_change_bottom8_bars + zero_share_rule
).properties(
    width=820,
    height=360,
    title=alt.TitleParams(
        'Change in income share by global decile, excluding deciles 9 and 10',
        subtitle=f'Change in percentage points, {START_YEAR}-{END_YEAR}. Dashed line: no change.',
    ),
).configure_view(stroke=None).configure_axis(
    grid=True,
    gridColor='#d9e2ec',
    gridDash=[3, 3],
)
share_change_bottom8_chart.save(str(DRAFT_DIR / '08_income_share_change_deciles_1_to_8.png'), scale_factor=2)
share_change_bottom8_chart

alt.LayerChart(...)

In [ ]:
# Idea: the global population distribution across welfare levels
distribution_years = [START_YEAR, 2000, 2010, 2020, END_YEAR]
distribution_data = data[data['year'].isin(distribution_years) & data['bin_pop'].notna()].copy()
distribution_bins = np.geomspace(
    distribution_data['welf_b'].min(),
    distribution_data['welf_b'].max(),
    80,
)
distribution_parts = []

for selected_year in distribution_years:
    year_data = distribution_data[distribution_data['year'] == selected_year]
    weighted_counts, _ = np.histogram(
        year_data['welf_b'],
        bins=distribution_bins,
        weights=year_data['bin_pop'],
    )
    distribution_parts.append(pd.DataFrame({
        'year': selected_year,
        'welfare_midpoint': np.sqrt(distribution_bins[:-1] * distribution_bins[1:]),
        'population_share': 100 * weighted_counts / weighted_counts.sum(),
    }))

global_distribution = pd.concat(distribution_parts, ignore_index=True)
distribution_lines = alt.Chart(global_distribution).mark_line(strokeWidth=2.5).encode(
    x=alt.X(
        'welfare_midpoint:Q',
        title='Daily welfare per person (2021 PPP dollars, log scale)',
        scale=alt.Scale(type='log'),
        axis=alt.Axis(format='$,.0f'),
    ),
    y=alt.Y('population_share:Q', title='Population in welfare range (%)'),
    color=alt.Color('year:O', title='Year', scale=alt.Scale(scheme='viridis')),
    detail='year:O',
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('welfare_midpoint:Q', title='Welfare level', format='$,.2f'),
        alt.Tooltip('population_share:Q', title='Population share (%)', format='.2f'),
    ],
)
global_distribution_chart = (
    distribution_lines
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'How the world is distributed across welfare levels',
            subtitle='Population-weighted distribution of daily welfare. Each curve sums to 100%.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
global_distribution_chart.save(str(FIG_DIR / 'global_welfare_distribution.png'), scale_factor=2)
global_distribution_chart

alt.Chart(...)

In [ ]:
# Visualization 9: the global population distribution in 1990 and 2020
distribution_1990_2020 = global_distribution[global_distribution['year'].isin([START_YEAR, 2020])].copy()
distribution_1990_2020_lines = alt.Chart(distribution_1990_2020).mark_line(strokeWidth=3).encode(
    x=alt.X(
        'welfare_midpoint:Q',
        title='Daily welfare per person (2021 PPP dollars, log scale)',
        scale=alt.Scale(type='log'),
        axis=alt.Axis(format='$,.0f'),
    ),
    y=alt.Y('population_share:Q', title='Population in welfare range (%)'),
    color=alt.Color('year:O', title='Year', scale=alt.Scale(domain=[1990, 2020], range=['#D55E00', '#0072B2'])),
    detail='year:O',
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('welfare_midpoint:Q', title='Welfare level', format='$,.2f'),
        alt.Tooltip('population_share:Q', title='Population share (%)', format='.2f'),
    ],
)
global_distribution_1990_2020_chart = (
    distribution_1990_2020_lines
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'How the world distribution shifted between 1990 and 2020',
            subtitle='Population-weighted distribution of daily welfare. Orange: 1990. Blue: 2020.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
global_distribution_1990_2020_chart.save(str(DRAFT_DIR / '09_global_welfare_distribution_1990_2020.png'), scale_factor=2)
global_distribution_1990_2020_chart

alt.Chart(...)

In [ ]:
# Idea: global income shares as stacked bars
share_bar_data = deciles[deciles['year'].isin([START_YEAR, 2000, 2010, 2020, END_YEAR])].copy()
share_bar_data['year_label'] = share_bar_data['year'].astype(str)
share_bar_data['decile_label'] = 'D' + share_bar_data['decile'].astype(str)

decile_order = [f'D{i}' for i in range(1, 11)]
decile_colors = [
    '#eff3ff', '#c6dbef', '#9ecae1', '#6baed6', '#4292c6',
    '#2171b5', '#08519c', '#08306b', '#041f4a', '#02132f',
]

share_bars = alt.Chart(share_bar_data).mark_bar().encode(
    x=alt.X('year_label:N', title=None, sort=['1990', '2000', '2010', '2020', '2026']),
    y=alt.Y('share:Q', title='Global income share (%)', stack='zero', scale=alt.Scale(domain=[0, 100])),
    order=alt.Order('decile:Q', sort='descending'),
    color=alt.Color(
        'decile_label:N',
        title='Global decile',
        sort=decile_order,
        scale=alt.Scale(domain=decile_order, range=decile_colors),
    ),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('decile_label:N', title='Decile'),
        alt.Tooltip('share:Q', title='Income share (%)', format='.2f'),
    ],
)
share_bars_chart = (
    share_bars.properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'Global income shares by decile',
            subtitle='Stacked bars show the composition of global income in selected years. D10 is darkest and forms the base.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
share_bars_chart.save(str(FIG_DIR / 'global_income_shares_stacked_bars.png'), scale_factor=2)


In [ ]:
# Idea: global income shares as stacked bars
display(share_bars_chart)

alt.Chart(...)

In [ ]:
# Visualization 10: D1 headcount compared with the $3 poverty line
poverty_line = 3

d1_counts = (
    data[data['decile'].eq(1)]
    .groupby('year', as_index=False)['bin_pop']
    .sum()
    .assign(group='D1: bottom 10%')
)
absolute_poverty_counts = (
    data[data['welf_b'].le(poverty_line)]
    .groupby('year', as_index=False)['bin_pop']
    .sum()
    .assign(group='Below $3/day')
)
comparison_counts = pd.concat([d1_counts, absolute_poverty_counts], ignore_index=True)
comparison_counts = comparison_counts[comparison_counts['year'].between(START_YEAR, END_YEAR)]
year_population = (
    data.groupby('year', as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'world_population'})
)
comparison_counts = comparison_counts.merge(year_population, on='year', how='left')
comparison_counts['people_millions'] = comparison_counts['bin_pop'] / 1e6
comparison_counts['population_share_pct'] = 100 * comparison_counts['bin_pop'] / comparison_counts['world_population']

latest_poverty_share = comparison_counts[
    (comparison_counts['year'] == END_YEAR) & (comparison_counts['group'] == 'Below $3/day')
]['population_share_pct'].iloc[0]
print(f'{END_YEAR}: {latest_poverty_share:.2f}% of the world population is below $3/day')

poor_headcount_chart = (
    alt.Chart(comparison_counts)
    .mark_line(point=True, strokeWidth=3)
    .encode(
        x=alt.X(
            'year:Q',
            title=None,
            axis=alt.Axis(format='d'),
            scale=alt.Scale(domain=[START_YEAR, END_YEAR]),
        ),
        y=alt.Y('people_millions:Q', title='People (millions)', scale=alt.Scale(zero=False)),
        color=alt.Color(
            'group:N',
            title='Measure',
            scale=alt.Scale(range=['#6baed6', '#d7301f']),
        ),
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('group:N', title='Measure'),
            alt.Tooltip('people_millions:Q', title='People (millions)', format=',.1f'),
            alt.Tooltip('population_share_pct:Q', title='Share of world population (%)', format='.2f'),
        ],
    )
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'How many people are in D1 or below $3 per day?',
            subtitle='D1 is relative: the bottom 10% globally. The $3 line is absolute, using welfare in PPP dollars per person per day.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(poor_headcount_chart)

2026: 9.91% of the world population is below $3/day


alt.Chart(...)

In [ ]:
# Idea: where do people in the poorest global decile live?
selected_poor_years = [START_YEAR, 2000, 2010, 2020, END_YEAR]
d1_country = (
    data[data['decile'].eq(1) & data['year'].isin(selected_poor_years)]
    .groupby(['year', 'code'], as_index=False)['bin_pop']
    .sum()
)
top_codes = (
    d1_country[d1_country['year'] == END_YEAR]
    .nlargest(15, 'bin_pop')['code']
    .tolist()
)
d1_country = d1_country[d1_country['code'].isin(top_codes)].copy()
d1_country['people_millions'] = d1_country['bin_pop'] / 1e6

d1_country_chart = (
    alt.Chart(d1_country)
    .mark_bar()
    .encode(
        x=alt.X('people_millions:Q', title='People in D1 (millions)'),
        y=alt.Y('code:N', title='Country code', sort=top_codes),
        color=alt.Color('year:O', title='Year', scale=alt.Scale(scheme='viridis')),
        yOffset='year:O',
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('people_millions:Q', title='People in D1 (millions)', format=',.1f'),
        ],
    )
    .properties(
        width=820,
        height=500,
        title=alt.TitleParams(
            'Where do people in the poorest global decile live?',
            subtitle='Top countries by number of people in D1 in 2026, shown in selected years.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(d1_country_chart)

alt.Chart(...)

In [ ]:
# Visualization 11: which countries have the largest share of their population in D1?
country_population = (
    data[data['year'] == END_YEAR]
    .groupby('code', as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'country_pop'})
)
country_d1_population = (
    data[(data['year'] == END_YEAR) & data['decile'].eq(1)]
    .groupby('code', as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'd1_pop'})
)
country_poverty_share = country_population.merge(country_d1_population, on='code', how='left').fillna({'d1_pop': 0})
country_poverty_share['d1_population_share'] = 100 * country_poverty_share['d1_pop'] / country_poverty_share['country_pop']
country_poverty_share['country_population_millions'] = country_poverty_share['country_pop'] / 1e6

country_poverty_share_chart = (
    alt.Chart(country_poverty_share)
    .mark_circle(opacity=0.75, stroke='#ffffff', strokeWidth=0.5)
    .encode(
        x=alt.X('country_population_millions:Q', title='Country population (millions)', scale=alt.Scale(type='log')),
        y=alt.Y('d1_population_share:Q', title='Share of country population in D1 (%)'),
        size=alt.Size('d1_pop:Q', title='People in D1', scale=alt.Scale(range=[30, 700])),
        color=alt.Color('d1_population_share:Q', title='D1 share (%)', scale=alt.Scale(scheme='reds')),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('country_population_millions:Q', title='Population (millions)', format=',.1f'),
            alt.Tooltip('d1_population_share:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('d1_pop:Q', title='People in D1', format=',.0f'),
        ],
    )
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'D1 concentration across countries',
            subtitle='Each point is a country in 2026. Size shows the number of people in D1; the x-axis is logarithmic.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(country_poverty_share_chart)

high_d1_countries = country_poverty_share[country_poverty_share['d1_population_share'] > 50].copy()
print(f"Countries with more than 50% of their population in D1: {len(high_d1_countries)}")

high_d1_chart = (
    alt.Chart(high_d1_countries)
    .mark_circle(opacity=0.8, stroke='#ffffff', strokeWidth=0.5)
    .encode(
        x=alt.X('country_population_millions:Q', title='Country population (millions)', scale=alt.Scale(type='log')),
        y=alt.Y('d1_population_share:Q', title='Share of country population in D1 (%)', scale=alt.Scale(domain=[50, 100])),
        size=alt.Size('d1_pop:Q', title='People in D1', scale=alt.Scale(range=[50, 800])),
        color=alt.Color('d1_population_share:Q', title='D1 share (%)', scale=alt.Scale(scheme='reds')),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('country_population_millions:Q', title='Population (millions)', format=',.1f'),
            alt.Tooltip('d1_population_share:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('d1_pop:Q', title='People in D1', format=',.0f'),
        ],
    )
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'Countries where more than half the population is in D1',
            subtitle='2026. Only countries with a D1 population share above 50% are shown.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(high_d1_chart)

alt.Chart(...)

Countries with more than 50% of their population in D1: 13


alt.Chart(...)

In [ ]:
# Idea: persistence of D1 concentration by country
selected_persistence_years = [START_YEAR, 2000, 2010, 2020, END_YEAR]
country_year_population = (
    data[data['year'].isin(selected_persistence_years)]
    .groupby(['year', 'code'], as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'country_pop'})
)
country_year_d1 = (
    data[data['year'].isin(selected_persistence_years) & data['decile'].eq(1)]
    .groupby(['year', 'code'], as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'd1_pop'})
)
persistence_data = country_year_population.merge(country_year_d1, on=['year', 'code'], how='left').fillna({'d1_pop': 0})
persistence_data['d1_population_share'] = 100 * persistence_data['d1_pop'] / persistence_data['country_pop']
top_persistent_codes = (
    persistence_data.groupby('code')['d1_population_share']
    .mean()
    .nlargest(20)
    .index
    .tolist()
)
persistence_data = persistence_data[persistence_data['code'].isin(top_persistent_codes)]

persistence_chart = (
    alt.Chart(persistence_data)
    .mark_rect()
    .encode(
        x=alt.X('year:O', title=None, sort=selected_persistence_years),
        y=alt.Y('code:N', title='Country code', sort=top_persistent_codes),
        color=alt.Color('d1_population_share:Q', title='Population in D1 (%)', scale=alt.Scale(scheme='reds')),
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('d1_population_share:Q', title='Population in D1 (%)', format='.1f'),
        ],
    )
    .properties(
        width=820,
        height=500,
        title=alt.TitleParams(
            'Where is D1 concentration persistent?',
            subtitle='Top 20 countries by average share of their population in the global poorest decile.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=False)
)
display(persistence_chart)

alt.Chart(...)

In [ ]:
# Idea: country welfare and concentration of D1
country_welfare = (
    data[data['year'] == END_YEAR]
    .groupby('code', as_index=False)
    .agg(country_pop=('bin_pop', 'sum'), welfare_x_pop=('welf_x_pop', 'sum'))
)
country_welfare['mean_welfare'] = country_welfare['welfare_x_pop'] / country_welfare['country_pop']
country_welfare = country_welfare.merge(
    country_poverty_share[['code', 'd1_pop', 'd1_population_share']],
    on='code',
    how='left',
)
country_welfare['country_population_millions'] = country_welfare['country_pop'] / 1e6

welfare_poverty_chart = (
    alt.Chart(country_welfare)
    .mark_circle(opacity=0.75, stroke='#ffffff', strokeWidth=0.5)
    .encode(
        x=alt.X('mean_welfare:Q', title='Mean welfare per person (2026 PPP dollars/day)', scale=alt.Scale(type='log')),
        y=alt.Y('d1_population_share:Q', title='Share of country population in D1 (%)'),
        size=alt.Size('country_pop:Q', title='Country population', scale=alt.Scale(range=[30, 700])),
        color=alt.Color('d1_population_share:Q', title='D1 share (%)', scale=alt.Scale(scheme='reds')),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('mean_welfare:Q', title='Mean welfare ($/day)', format='$,.2f'),
            alt.Tooltip('d1_population_share:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('country_population_millions:Q', title='Population (millions)', format=',.1f'),
        ],
    )
    .properties(
        width=820,
        height=420,
        title=alt.TitleParams(
            'Countries with lower welfare tend to concentrate more people in D1',
            subtitle='Each point is a country in 2026. Size shows population; both axes use welfare-related measures.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(welfare_poverty_chart)

alt.Chart(...)

In [ ]:
# Idea: D1 concentration and poverty share in 1990
snapshot_year = START_YEAR
snapshot_data = data[data['year'] == snapshot_year].copy()

snapshot_country = (
    snapshot_data.groupby('code', as_index=False)
    .agg(country_pop=('bin_pop', 'sum'))
)
snapshot_d1 = (
    snapshot_data[snapshot_data['decile'] == 1]
    .groupby('code', as_index=False)
    .agg(d1_pop=('bin_pop', 'sum'))
)
snapshot_below_3 = (
    snapshot_data[snapshot_data['welf_b'] <= poverty_line]
    .groupby('code', as_index=False)
    .agg(below_3_pop=('bin_pop', 'sum'))
)

snapshot_country = (
    snapshot_country
    .merge(snapshot_d1, on='code', how='left')
    .merge(snapshot_below_3, on='code', how='left')
    .fillna({'d1_pop': 0, 'below_3_pop': 0})
)
snapshot_country['d1_share_pct'] = 100 * snapshot_country['d1_pop'] / snapshot_country['country_pop']
snapshot_country['below_3_share_pct'] = 100 * snapshot_country['below_3_pop'] / snapshot_country['country_pop']
snapshot_country['country_population_millions'] = snapshot_country['country_pop'] / 1e6

snapshot_correlation = snapshot_country[
    ['d1_share_pct', 'below_3_share_pct']
].corr().iloc[0, 1]
print(f'Correlation in {snapshot_year}: {snapshot_correlation:.2f}')

snapshot_points = (
    alt.Chart(snapshot_country)
    .mark_circle(opacity=0.8, stroke='#ffffff', strokeWidth=0.5)
    .encode(
        x=alt.X('d1_share_pct:Q', title='Population in D1 (%)', scale=alt.Scale(zero=True)),
        y=alt.Y('below_3_share_pct:Q', title='Population below $3 (%)', scale=alt.Scale(zero=True)),
        size=alt.Size(
            'country_pop:Q',
            title='Country population',
            scale=alt.Scale(range=[30, 700]),
        ),
        color=alt.Color(
            'below_3_share_pct:Q',
            title='Below $3 (%)',
            scale=alt.Scale(scheme='reds'),
        ),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('d1_share_pct:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('below_3_share_pct:Q', title='Population below $3 (%)', format='.1f'),
            alt.Tooltip('country_population_millions:Q', title='Population (millions)', format=',.1f'),
        ],
    )
)
snapshot_labels = (
    alt.Chart(snapshot_country)
    .mark_text(align='left', dx=7, dy=-7, fontSize=10)
    .encode(
        x='d1_share_pct:Q',
        y='below_3_share_pct:Q',
        text='code:N',
    )
)
snapshot_1990_chart = (
    (snapshot_points + snapshot_labels)
    .properties(
        width=820,
        height=500,
        title=alt.TitleParams(
            'D1 concentration and poverty rates across countries in 1990',
            subtitle='Each point is a country. D1 is relative to the global distribution; $3 is an absolute PPP welfare threshold.',
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(snapshot_1990_chart)

Correlation in 1990: 0.85


alt.LayerChart(...)

In [ ]:
# Table and chart: changes over time by D1 population-share threshold
comparison_years = [1990, 2000, 2010, 2020, 2026]
thresholds = list(range(10, 100, 10))
threshold_labels = [f'>{threshold}%' for threshold in thresholds]

country_population = (
    data.groupby(['year', 'code'], as_index=False)
    .agg(country_pop=('bin_pop', 'sum'))
)
d1_population = (
    data[data['decile'] == 1]
    .groupby(['year', 'code'], as_index=False)
    .agg(d1_pop=('bin_pop', 'sum'))
)

country_d1_shares = country_population.merge(
    d1_population,
    on=['year', 'code'],
    how='left',
).fillna({'d1_pop': 0})
country_d1_shares['d1_share_pct'] = (
    100 * country_d1_shares['d1_pop'] / country_d1_shares['country_pop']
)

country_counts_table = pd.DataFrame({
    f'>{threshold}%': (
        country_d1_shares[
            (country_d1_shares['year'].isin(comparison_years))
            & (country_d1_shares['d1_share_pct'] > threshold)
        ]
        .groupby('year')
        .size()
        .reindex(comparison_years, fill_value=0)
        .to_numpy()
    )
    for threshold in thresholds
}, index=comparison_years).rename_axis('Year').reset_index()

display(country_counts_table)


# For 2026, these three countries have more than 80% of their population in D1
##Republic of South Sudan = SSD
##Democratic Republic of the Congo = COD
##Mozambique = MOZ

,Year,>10%,>20%,>30%,>40%,>50%,>60%,>70%,>80%,>90%
0,1990,43,23,16,8,8,4,1,1,1
1,2000,57,31,20,11,7,4,2,1,0
2,2010,58,38,22,14,7,6,2,1,0
3,2020,56,41,33,22,12,10,5,2,0
4,2026,54,44,34,24,13,10,7,3,0


In [ ]:
# Save Visualization 11 for use in the milestone draft
poor_headcount_chart.save(str(DRAFT_DIR / '10_d1_headcount_vs_3_dollars.png'), scale_factor=2)
poor_headcount_chart

alt.Chart(...)

In [ ]:
# Save Visualization 12 for use in the milestone draft
country_poverty_share_chart.save(str(DRAFT_DIR / '11_d1_concentration_across_countries.png'), scale_factor=2)
country_poverty_share_chart

alt.Chart(...)